In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

¡CONECTADO! Título de la página: Google


In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
# options.binary_location = "/usr/bin/google-chrome" # Solo si es necesario en tu entorno
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --- PASO 2: VARIABLES DE BÚSQUEDA (MODIFICAR AQUÍ) ---
URL_OBJETIVO = "https://www.python.org/jobs/"
SELECTOR_CONTENEDOR = "ol.list-recent-jobs li" # El bloque que envuelve cada trabajo
SELECTOR_TITULO = ".listing-company-name"      # El nombre del cargo
SELECTOR_INFO = ".listing-job-type"            # Ubicación o tipo de contrato

paginas_objetivo = 3 # Número de páginas a recorrer
dataset_final = []

try:
    driver.get(URL_OBJETIVO)
    
    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")
        
        # Esperar a que cargue la lista de empleos
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
        )

        # CAPTURA DE DATOS
        empleos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
        
        for item in empleos:
            try:
                # Extraemos el texto de los elementos internos
                nombre = item.find_element(By.CSS_SELECTOR, SELECTOR_TITULO).text
                detalle = item.find_element(By.CSS_SELECTOR, SELECTOR_INFO).text
                
                dataset_final.append({
                    "Puesto": nombre.replace("\n", " ").strip(),
                    "Detalle": detalle.strip(),
                    "Pagina_Origen": i + 1
                })
            except:
                continue

        # NAVEGACIÓN: Clic en el botón "Next"
        try:
            # Buscamos el botón que contenga la palabra "Next"
            boton_siguiente = driver.find_element(By.XPATH, "//li[@class='next']/a")
            
            # Scroll hasta el botón para asegurar que sea clickeable
            driver.execute_script("arguments[0].scrollIntoView();", boton_siguiente)
            time.sleep(1) 
            
            boton_siguiente.click()
            print("Cambiando de página...")
            time.sleep(3) # Pausa para renderizado
            
        except Exception as e:
            print("No hay más páginas disponibles o error en XPATH.")
            break

    # --- PASO 4: SALIDA DE DATOS ---
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("empleos_python_org.csv", index=False)
        print(f"\nProceso exitoso. Registros capturados: {len(df)}")
        print(df.head())
    else:
        print("No se capturaron datos.")

finally:
    driver.quit()
    print("Navegador cerrado.")

--- Procesando Página 1 ---
No hay más páginas disponibles o error en XPATH.

Proceso exitoso. Registros capturados: 19
                                              Puesto  \
0  NEW Senior Python Developer - full stack Trust...   
1  NEW Senior Research Software Engineer (Multiph...   
2  NEW Senior Research Software Engineer (Neural ...   
3        NEW Python Software Engineer HypothesisBase   
4  NEW Python Developer - Remote DivIHN Integrati...   

                                             Detalle  Pagina_Origen  
0  Back end, Big Data, Cloud, Database, Machine L...              1  
1                         Big Data, Image Processing              1  
2  Big Data, Database, Image Processing, Numeric ...              1  
3       Back end, Integration, Systems, Testing, Web              1  
4                      Cloud, Web, Support, Security              1  
Navegador cerrado.


In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# Añadimos un tamaño de ventana para que el botón sea visible
options.add_argument("--window-size=1920,1080") 

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --- PASO 2: VARIABLES ---
URL_OBJETIVO = "https://www.python.org/jobs/"
SELECTOR_CONTENEDOR = "ol.list-recent-jobs li"
SELECTOR_TITULO = ".listing-company-name"
SELECTOR_INFO = ".listing-job-type"

paginas_objetivo = 3 # Cambia a 5 si necesitas más
dataset_final = []

try:
    driver.get(URL_OBJETIVO)
    
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")
        
        # Espera a que los empleos sean visibles
        WebDriverWait(driver, 10).until(
            EC.visibility_of_any_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
        )

        # CAPTURA
        empleos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
        for item in empleos:
            try:
                nombre = item.find_element(By.CSS_SELECTOR, SELECTOR_TITULO).text
                detalle = item.find_element(By.CSS_SELECTOR, SELECTOR_INFO).text
                
                dataset_final.append({
                    "Puesto": nombre.replace("\n", " ").strip(),
                    "Detalle": detalle.strip(),
                    "Pagina": i + 1
                })
            except:
                continue

        # --- NAVEGACIÓN (EL CAMBIO CLAVE AQUÍ) ---
        if i < paginas_objetivo - 1: # Solo busca el botón si no es la última página
            try:
                # XPATH que busca específicamente el enlace dentro del LI con clase 'next'
                boton_siguiente = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.XPATH, "//li[@class='next']/a"))
                )
                
                # Scroll suave hasta el botón
                driver.execute_script("arguments[0].scrollIntoView();", boton_siguiente)
                time.sleep(1)
                
                # Clic mediante JavaScript (más confiable para botones dinámicos)
                driver.execute_script("arguments[0].click();", boton_siguiente)
                
                print(" Navegando a la siguiente página...")
                time.sleep(2) # Espera para carga de red
            except:
                print(" No se encontró el botón 'Next' o es la última página.")
                break

    # --- SALIDA ---
    if dataset_final:
        df = pd.DataFrame(dataset_final)
        df.to_csv("datos_python_jobs.csv", index=False)
        print(f"\n¡Éxito! Registros totales: {len(df)}")
        print(df.tail()) # Muestra los últimos para ver si cambió de página
    else:
        print("Error: No se capturaron datos.")

finally:
    driver.quit()

--- Procesando Página 1 ---
 No se encontró el botón 'Next' o es la última página.

¡Éxito! Registros totales: 19
                                               Puesto  \
14  Junior Full Stack Developer Patrick J. McGover...   
15       Senior Python Backend Developer SKYCATCHFIRE   
16                    Senior Software Engineer Aquent   
17  Senior Data Privacy Software Developer OpenDP ...   
18  Senior Backend Python Developer Kanmalai Techn...   

                                              Detalle  Pagina  
14                                Back end, Front end       1  
15                                           Back end       1  
16  Back end, Cloud, Front end, Machine Learning, Web       1  
17               Machine Learning, Numeric processing       1  
18                                           Back end       1  
